In [ ]:
import os
import json
import fitz
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from sentence_transformers import SentenceTransformer
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document


print("All libraries loaded!")

c:\Users\azbas\Downloads\rag-financial-engine\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries loaded!


C:\Users\azbas\AppData\Local\Temp\ipykernel_13164\1679954758.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


In [2]:
### pdf parsing

def extract_text_from_pdf(pdf_path):
    """Extract text from PDF using PyMuPDF"""
    doc = fitz.open(pdf_path)
    text = ""
    for page_num in range(len(doc)):
        page = doc[page_num]
        text += page.get_text()
    doc.close()
    return text

def chunk_text(text, chunk_size=512, overlap=100):
    """Split text into chunks"""
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
    
    return chunks

def process_all_pdfs():
    """Process all PDFs in data/reports/"""
    reports_dir = "data/reports"
    
    if not os.path.exists(reports_dir):
        print(f"Error: {reports_dir} folder not found!")
        return
    
    all_chunks = []
    
    for pdf_file in os.listdir(reports_dir):
        if pdf_file.endswith('.pdf'):
            pdf_path = os.path.join(reports_dir, pdf_file)
            company_name = pdf_file.replace('.pdf', '').replace('2025_2026', '')
            
            print(f"Processing: {pdf_file}...")
            
            text = extract_text_from_pdf(pdf_path)
            
            chunks = chunk_text(text)
            
            for i, chunk in enumerate(chunks):
                chunk_data = {
                    "company": company_name,
                    "pdf_file": pdf_file,
                    "chunk_id": i,
                    "text": chunk,
                    "tokens": len(chunk.split())
                }
                all_chunks.append(chunk_data)
            
            print(f"  → Created {len(chunks)} chunks from {company_name}")
    
    # Save chunks to JSON
    output_file = "data/chunks.json"
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(all_chunks, f, indent=2, ensure_ascii=False)
    
    print(f"\nTotal chunks created: {len(all_chunks)}")
    print(f"Chunks saved to: {output_file}")
    return all_chunks

chunks = process_all_pdfs()

Processing: ADANIPOWER2025_2026.pdf...
  → Created 621 chunks from ADANIPOWER
Processing: AMBUJACEMENT2025_2026.pdf...
  → Created 700 chunks from AMBUJACEMENT
Processing: ASIANPAINTS2025_2026.pdf...
  → Created 576 chunks from ASIANPAINTS
Processing: BAJAJAUTO2025_2026.pdf...
  → Created 421 chunks from BAJAJAUTO
Processing: CIPLA2025_2026.pdf...
  → Created 563 chunks from CIPLA
Processing: COLGATE2025_2026.pdf...
  → Created 301 chunks from COLGATE
Processing: INFOSYS2025_2026.pdf...
  → Created 471 chunks from INFOSYS
Processing: ITC2025_2026.pdf...
  → Created 488 chunks from ITC
Processing: JSWSTEEL2025_2026.pdf...
  → Created 870 chunks from JSWSTEEL
Processing: M&M2025_2026.pdf...
  → Created 580 chunks from M&M
Processing: RELIANCE2025_2026.pdf...
  → Created 416 chunks from RELIANCE
Processing: SUNPHARMA2025_2026.pdf...
  → Created 391 chunks from SUNPHARMA
Processing: TATAPOWER2025_2026.pdf...
  → Created 742 chunks from TATAPOWER
Processing: TATASTEEL2025_2026.pdf...
  → Cr

In [3]:
# Load chunks from JSON
with open('data/chunks.json', 'r', encoding='utf-8') as f:
    chunks_data = json.load(f)

print(f"Loaded {len(chunks_data)} chunks")
print(f"First chunk text sample: {chunks_data[0]['text'][:100]}...")

Loaded 10675 chunks
First chunk text sample: Adani Power Limited Tel +91 79 2656 7555 “Adani Corporate House” Fax +91 79 2555 7177 Shantigram, Ne...


In [8]:
### Embedding


print("Loading Hugging Face Sentence Transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding all chunks (this will take a few minutes)...")
texts = [chunk['text'] for chunk in chunks_data]
embeddings = model.encode(texts, show_progress_bar=True, batch_size=32)

print(f"Embeddings created: {embeddings.shape}")

# Add embeddings to chunks
for i, chunk in enumerate(chunks_data):
    chunk['embedding'] = embeddings[i].tolist()

print(f"Total chunks with embeddings: {len(chunks_data)}")

Loading Hugging Face Sentence Transformer model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2786.95it/s]


Embedding all chunks (this will take a few minutes)...


Batches: 100%|██████████| 334/334 [08:22<00:00,  1.50s/it]


Embeddings created: (10675, 384)
Total chunks with embeddings: 10675


In [13]:
### store data in chromadb


documents = []
for chunk in chunks_data:
    doc = Document(
        page_content=chunk['text'],
        metadata={
            'company': chunk['company'],
            'pdf_file': chunk['pdf_file'],
            'chunk_id': chunk['chunk_id']
        }
    )
    documents.append(doc)

print(f"Created {len(documents)} documents for Chroma")

print("Initializing Chroma vector database...")
embeddings_model = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings_model,
    persist_directory="./chroma_db"
)

print(f"Chroma vector database created!")
print(f"Stored {len(documents)} documents in Chroma")

Created 10675 documents for Chroma
Initializing Chroma vector database...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1908.66it/s]


Chroma vector database created!
Stored 10675 documents in Chroma


In [14]:
# Test retrieval
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a retrieval-augmented financial analysis assistant.

Your ONLY source of truth is the retrieved context provided below.

Rules:

1. Answer ONLY using information explicitly present in the retrieved context.
2. Never use your own knowledge, memory, assumptions, or outside information.
3. If the answer cannot be found completely in the retrieved context, respond exactly:

"The retrieved documents do not contain enough information to answer this question."

4. Never invent:
- financial figures
- dates
- percentages
- company names
- business strategies
- risks
- conclusions
- calculations that are not explicitly supported.

5. If multiple retrieved chunks contain conflicting information:
- mention the conflict.
- do not choose one unless the context clearly indicates which is correct.

6. When answering numerical questions:
- copy numbers exactly as written.
- preserve units (₹, $, million, billion, %, etc.)
- never round unless the document already does.

7. If the user asks for a comparison:
- compare only companies or values present in the retrieved context.
- if one side of the comparison is missing, state that there is insufficient information.

8. If the user asks for a summary:
- summarize only what appears in the retrieved context.
- do not add explanations or external background knowledge.

9. If the user asks "why", "what caused", or requests an interpretation:
- answer only if the retrieved text explicitly states the reason.
- otherwise say the documents do not specify the reason.

10. Do not speculate.
11. Do not guess.
12. Be concise and factual.
13. If possible, cite which retrieved document or company the information came from using the provided metadata.

Context:
{context}

Question:
{question}

Answer:
"""
)

print("Testing retrieval...\n")

query = "What is the company's revenue?"
results = vectorstore.similarity_search(query, k=5)

print(f"Retrieved {len(results)} chunks for: '{query}'\n")

for i, result in enumerate(results, 1):
    print(f"Result {i}: {result.metadata['company'].upper()}")
    print(f"  Text: {result.page_content[:100]}...\n")

print("Testing retrieval...\n")
query = "What is the company's revenue?"
results = vectorstore.similarity_search(query, k=5)

print(f"Retrieved {len(results)} chunks for: '{query}'\n")
for i, result in enumerate(results, 1):
    print(f"Result {i}: {result.metadata['company'].upper()}")
    print(f"  Text: {result.page_content[:100]}...\n")

Testing retrieval...

Retrieved 5 chunks for: 'What is the company's revenue?'

Result 1: INFOSYS2025_2026
  Text: Group presents revenues net of indirect taxes in its Consolidated Statement of Profit and Loss. Reve...

Result 2: INFOSYS2025_2026
  Text: Group presents revenues net of indirect taxes in its Consolidated Statement of Profit and Loss. Reve...

Result 3: INFOSYS
  Text: Group presents revenues net of indirect taxes in its Consolidated Statement of Profit and Loss. Reve...

Result 4: INFOSYS
  Text: Group presents revenues net of indirect taxes in its Consolidated Statement of Profit and Loss. Reve...

Result 5: TCS
  Text: 3,200 2,955 Change during the year (net) 3,112 245 6,312 3,200 1,06,878 94,394 11) Other Equity Stat...

Testing retrieval...

Retrieved 5 chunks for: 'What is the company's revenue?'

Result 1: INFOSYS2025_2026
  Text: Group presents revenues net of indirect taxes in its Consolidated Statement of Profit and Loss. Reve...

Result 2: INFOSYS2025_2026
  Te

In [15]:
# trying to connect Grok to the retrieval system
# get question -> retrieve chunks -> prepare for Grok

import os
from dotenv import load_dotenv

load_dotenv()

# check if grok key is there
grok_key = os.getenv('GROK_API_KEY')
print(f"Grok key: {grok_key[:10]}..." if grok_key else "No grok key found!")

# function to retrieve and prepare context
def prepare_for_grok(question):
    # retrieve chunks
    results = vectorstore.similarity_search(question, k=5)
    
    # combine into context
    context = "\n".join([result.page_content for result in results])
    
    print(f"Q: {question}")
    print(f"Retrieved {len(results)} chunks")
    print(f"\nContext prepared for Grok:")
    print(context[:300])
    
    return context, results

print("Ready to send to Grok")

Grok key: xai-w5xloP...
Ready to send to Grok


In [18]:
context, docs = prepare_for_grok("What is adani revenue?")

Q: What is adani revenue?
Retrieved 5 chunks

Context prepared for Grok:
Enterprises Limited 532.87 - 532.87 - 15 Loans interest - Repaid Adani Enterprises Limited 20.50 - 20.50 - 719 718 Integrated Annual Report 2025-26 AMBUJA CEMENTS LIMITED Portfolio Overview Corporate Overview Strategic Review ESG Overview Statutory Reports Financial Statements Notes to Consolidated 


In [ ]:
import sys
print(sys.executable)

In [6]:
print(f"Total chunks: {len(chunks_data)}")
print(f"Unique chunks:{len(set(chunk['text'] for chunk in chunks_data))}")

Total chunks: 10675
Unique chunks:10675
